# NB15 — Ablation Table + Final Report Figures

**Goal (HANDOFF §13 item 15):** Consolidated ablation table and four report-ready figures.
- Ablation bar chart: classifier-alone → +streaming → +VAE → +GAN
- Confusion matrix (classifier-alone, val set)
- Conti_06 entropy/confidence trajectory
- Cross-family bar chart (LOFO per-family)

## 1. Setup

In [ ]:
import numpy as np
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torch.nn as nn

SANDBOX = '/sessions/gallant-practical-davinci/mnt/Directed Study/Ransomware'
FEAT    = Path(SANDBOX) / 'features'
DEVICE  = torch.device('cpu')

CLASS_ORDER     = ['Benign', 'Legit-Encryption', 'Ransomware']
BENIGN_IDX, LENC_IDX, RAN_IDX = 0, 1, 2
VARIANT_CLASSES = [f'Conti_0{i}' for i in range(1, 8)]

split       = np.load(FEAT / 'fewshot_split_cnn.npz', allow_pickle=True)
val_idx     = split['val_idx']
test_idx    = split['test_idx']
labels_3way = split['labels_3way']

roc = json.load(open(FEAT / 'roc_analysis_results.json'))
gan = json.load(open(FEAT / 'gan_results.json'))
v2  = json.load(open(FEAT / 'fusion_v2_results.json'))
streaming_max = {int(k): float(v) for k, v in v2['streaming_max'].items()}
eval_idx = np.concatenate([val_idx, test_idx])

print("Data loaded.")

In [ ]:
class TextEmbeddingClassifier(nn.Module):
    def __init__(self, emb_dim=384, n_classes=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(emb_dim, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU())
        self.classifier    = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, n_classes))
        self.reconstructor = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, emb_dim))
    def forward(self, x):
        f = self.conv(x.permute(0, 2, 1)); p = f.mean(dim=-1)
        return self.classifier(p), self.reconstructor(p), p

clf = TextEmbeddingClassifier().to(DEVICE)
clf.load_state_dict(torch.load(FEAT / 'classifier.pt', map_location=DEVICE, weights_only=False))
clf.eval()
X_emb = np.load(FEAT / 'text_sequences.npy')

primary_probs = {}
with torch.no_grad():
    for i in eval_idx:
        x = torch.FloatTensor(X_emb[i]).unsqueeze(0)
        logits, _, _ = clf(x)
        primary_probs[i] = torch.softmax(logits, dim=1).cpu().numpy()[0]
print(f"Classifier loaded. Val accuracy: {sum(CLASS_ORDER[primary_probs[i].argmax()] == (labels_3way[i] if labels_3way[i] in CLASS_ORDER else 'Ransomware') for i in val_idx) / len(val_idx):.3f}")

## 2. Ablation data

In [ ]:
# Data from Results.md / previously run notebooks
clf_recall  = roc['classifier_alone']['var_recall']
fus_recall  = roc['operating_point']['var_recall']
fus_fpr     = roc['operating_point']['fpr_combined']
vae_all     = sum(v2['variant_recall'][v]['v2'] for v in VARIANT_CLASSES) / 70
vae_fpr_ben = v2['val_fpr']['Benign']['v2'] / v2['val_fpr']['Benign']['n']
vae_fpr_lnc = v2['val_fpr']['Legit-Encryption']['v2'] / v2['val_fpr']['Legit-Encryption']['n']
vae_fpr_comb = (v2['val_fpr']['Benign']['v2'] + v2['val_fpr']['Legit-Encryption']['v2']) / 12
gan_delta   = gan['variant_recall_gan_augmented'] - gan['variant_recall_real_only']
gan_recall  = clf_recall + gan_delta

c06_base = roc['classifier_alone']['conti_06_recall']
c06_fus  = roc['operating_point']['conti_06_recall']
c06_vae  = v2['variant_recall']['Conti_06']['v2'] / 10

LOFO = {
    'WannaCry':  {'static': 0.30, 'streaming': 1.00},
    'Ryuk':      {'static': 0.00, 'streaming': 0.10},
    'REvil':     {'static': 1.00, 'streaming': 1.00},
    'LockBit':   {'static': 0.70, 'streaming': 1.00},
    'Darkside':  {'static': 0.00, 'streaming': 0.90},
    'Conti':     {'static': 1.00, 'streaming': 1.00},
}

print(f"Ablation data computed.")
print(f"  Classifier-alone:  {clf_recall:.1%}  Conti_06={c06_base:.1%}  FPR=0%")
print(f"  Fusion-v1:         {fus_recall:.1%}  Conti_06={c06_fus:.1%}  FPR={fus_fpr:.1%}")
print(f"  Fusion-v2 (VAE):   {vae_all:.1%}  Conti_06={c06_vae:.1%}  FPR={vae_fpr_comb:.1%}")
print(f"  +GAN (head):       {gan_recall:.1%}  (head-only ablation; +{gan_delta:.1%} on baseline)")
print(f"  LOFO: static={sum(LOFO[f]['static'] for f in LOFO)/6:.1%}  streaming={sum(LOFO[f]['streaming'] for f in LOFO)/6:.1%}")

## 3. Ablation bar chart

In [ ]:
stages   = ['Classifier\nalone', '+Streaming\n(Fusion-v1)', '+VAE\n(Fusion-v2)', '+GAN\n(augmented)']
recalls  = [clf_recall, fus_recall, vae_all, gan_recall]
fprs     = [0.0,        fus_fpr,   vae_fpr_comb, 0.0]
colors_r = ['#3cb44b',  '#e6194b', '#f58231',  '#4363d8']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
bars = ax.bar(stages, [r*100 for r in recalls], color=colors_r, alpha=0.85, edgecolor='k', lw=0.5)
for bar, r in zip(bars, recalls):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{r:.1%}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Variant recall (%)'); ax.set_title('Within-family variant recall\nablation (70 Conti variants)')
ax.set_ylim(0, 108); ax.axhline(100, color='gray', lw=1, ls='--', alpha=0.4); ax.grid(True, alpha=0.2, axis='y')

ax2 = axes[1]
bars2 = ax2.bar(stages, [f*100 for f in fprs], color=colors_r, alpha=0.85, edgecolor='k', lw=0.5)
for bar, f in zip(bars2, fprs):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{f:.1%}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('FPR — non-ransomware combined (%)'); ax2.set_title('False-positive rate by component')
ax2.set_ylim(0, 50); ax2.grid(True, alpha=0.2, axis='y')

fig.text(0.5, -0.04, '*VAE: two-signal rule weaker at N=96; +GAN: head-only ablation, no fusion',
         ha='center', fontsize=8.5, style='italic', color='gray')
plt.tight_layout()
plt.savefig(FEAT / 'ablation_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ablation_chart.png")

## 4. Confusion matrix (classifier-alone, val set)

In [ ]:
val_true_3 = np.array([l if l in CLASS_ORDER else 'Ransomware' for l in labels_3way[val_idx]])
val_pred_3 = np.array([CLASS_ORDER[primary_probs[i].argmax()] for i in val_idx])
cm = np.zeros((3,3), dtype=int)
for t,p in zip(val_true_3, val_pred_3):
    cm[CLASS_ORDER.index(t), CLASS_ORDER.index(p)] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues'); plt.colorbar(im, ax=ax)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
short = ['Benign','Legit-Enc','Ransomware']
ax.set_xticklabels(short); ax.set_yticklabels(short)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix — Classifier-alone\n(val set, 24 execs)')
for i in range(3):
    for j in range(3):
        c = 'white' if cm[i,j] > cm.max()*0.5 else 'black'
        ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=14, color=c, fontweight='bold')
plt.tight_layout(); plt.savefig(FEAT / 'confusion_matrix_val.png', dpi=150); plt.show()

print(f"Val accuracy: {(val_pred_3==val_true_3).mean():.3f}")
print(f"1 Legit-Enc misclassified as Benign -> source of the Legit-Enc streaming FPR")

## 5. Conti_06 streaming trajectory

In [ ]:
c06_idx = test_idx[labels_3way[test_idx] == 'Conti_06']
W, STRIDE, NW = 50, 5, 300
n_pos = len(range(0, NW-W+1, STRIDE))
time_ax = np.array([start*0.1 + W*0.05 for start in range(0, NW-W+1, STRIDE)])

c06_traj = {}
ref_b = val_idx[labels_3way[val_idx]=='Benign'][0]
ref_l = val_idx[labels_3way[val_idx]=='Legit-Encryption'][0]
ref_r = val_idx[labels_3way[val_idx]=='Ransomware'][0]

def traj(i):
    p = np.zeros(n_pos)
    with torch.no_grad():
        for pos, st in enumerate(range(0, NW-W+1, STRIDE)):
            x = torch.FloatTensor(X_emb[i, st:st+W]).unsqueeze(0)
            logits,_,_ = clf(x)
            p[pos] = torch.softmax(logits, dim=1).cpu().numpy()[0, RAN_IDX]
    return p

for i in c06_idx: c06_traj[i] = traj(i)
rb, rl, rr = traj(ref_b), traj(ref_l), traj(ref_r)

fig, ax = plt.subplots(figsize=(12, 5))
caught = [i for i in c06_idx if streaming_max[i]>=0.70]
missed = [i for i in c06_idx if streaming_max[i]<0.70]
for i in caught:
    ax.plot(time_ax, c06_traj[i], color='#e6194b', alpha=0.55, lw=1.5,
            label='Conti_06 (caught)' if i==caught[0] else '')
for i in missed:
    ax.plot(time_ax, c06_traj[i], color='#f58231', alpha=0.55, lw=1.5, ls='--',
            label='Conti_06 (missed)' if i==missed[0] else '')
ax.plot(time_ax, rr, color='darkred',  lw=2, ls=':', label='Ransomware ref')
ax.plot(time_ax, rl, color='#4363d8',  lw=2, ls=':', label='Legit-Enc ref')
ax.plot(time_ax, rb, color='gray',     lw=2, ls=':', label='Benign ref')
ax.axhline(0.70, color='black', lw=1.5, ls='--', alpha=0.8, label='Threshold 0.70')
ax.set_xlabel('Window center time (s)'); ax.set_ylabel('P(Ransomware) in W=50 window')
ax.set_title('Conti_06 streaming trajectory\nEntropy transition: moderate (early) -> high (encryption onset)')
ax.legend(fontsize=9, loc='upper left')
ax.set_xlim(0,30); ax.set_ylim(-0.02,1.05); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(FEAT/'conti06_trajectory.png', dpi=150); plt.show()
print("Saved: conti06_trajectory.png")

## 6. Cross-family bar chart (LOFO)

In [ ]:
families = list(LOFO.keys())
statics  = [LOFO[f]['static']*100    for f in families]
streams  = [LOFO[f]['streaming']*100 for f in families]
x = np.arange(len(families)); w = 0.35

fig, ax = plt.subplots(figsize=(10,5))
b1 = ax.bar(x-w/2, statics, w, label='Static (classifier)', color='#3cb44b', alpha=0.85, edgecolor='k', lw=0.5)
b2 = ax.bar(x+w/2, streams, w, label='+ Streaming', color='#e6194b', alpha=0.85, edgecolor='k', lw=0.5)
for bar,v in zip(b1,statics): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{v:.0f}%', ha='center', fontsize=9)
for bar,v in zip(b2,streams): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{v:.0f}%', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(families)
ax.set_ylabel('Detection rate (%)'); ax.set_title('Cross-family zero-day recall (LOFO NB11)\nHeld-out family: 10 execs each, never in training')
ax.legend(); ax.set_ylim(0,115); ax.axhline(100,color='gray',lw=1,ls='--',alpha=0.4); ax.grid(True,alpha=0.2,axis='y')
ryuk_x = families.index('Ryuk')
ax.annotate('Oldest family\n(2018)', xy=(ryuk_x+w/2, 10), xytext=(ryuk_x+1, 30),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=8.5, color='gray')
plt.tight_layout(); plt.savefig(FEAT/'lofo_crossfamily.png', dpi=150); plt.show()
print("Saved: lofo_crossfamily.png")
print(f"\nMean: static={sum(statics)/6:.1f}%  streaming={sum(streams)/6:.1f}%")

## 7. Summary

In [ ]:
print("=" * 70)
print("ABLATION TABLE SUMMARY")
print("=" * 70)
print(f"{'Component':<26} {'Recall':>10} {'Conti_06':>10} {'FPR_comb':>10}")
print("-" * 60)
rows = [
    ("Classifier-alone (NB07b)",   clf_recall,  c06_base, 0.0),
    ("+Streaming (Fusion-v1)",     fus_recall,  c06_fus,  fus_fpr),
    ("+VAE (Fusion-v2, NB13)",     vae_all,     c06_vae,  vae_fpr_comb),
    ("+GAN head (NB12)",           gan_recall,  0.20,     0.0),
    ("BEST (Fusion-v1 @ 0.65)",    1.0,         1.0,      fus_fpr),
]
for r in rows:
    print(f"  {r[0]:<24} {r[1]:>10.1%} {r[2]:>10.1%} {r[3]:>10.1%}")
print(f"\nCross-family LOFO: static=50%, streaming=83.3%, FPR proxy<=25%")
print(f"Report optimal threshold: 0.65 (100% within-family, same 25% FPR as 0.70)")
print()
print("Figures saved:")
for f in ['ablation_chart.png','confusion_matrix_val.png','conti06_trajectory.png','lofo_crossfamily.png']:
    print(f"  features/{f}")